In [1]:
from datasets import load_dataset

dataset = load_dataset("imagenet-1k", split="validation", streaming=True)
dataset = dataset.take(1000)

In [2]:
import torch
import time
from torchvision import transforms
from tqdm import tqdm

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Baseline inference

In [4]:
def benchmark(model, dataset, num_samples=1000, batch_size=32, device='cpu'):
    # ImageNet transforms
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
    ])

    model.eval()
    model = model.to(device)

    correct = 0
    total = 0
    inference_times = []

    print("Evaluating model...")

    start_total = time.time()
    
    # Process samples in batches
    batch_images = []
    batch_labels = []
    
    with torch.no_grad():
        # Model warmup
        for i in range(10):
            random_input = torch.randn(1, 3, 224, 224).to(device)
            model(random_input)

        for i, item in enumerate(tqdm(dataset, total=num_samples, desc="Processing")):
            if i >= num_samples:
                break
            
            # Transform image
            image = transform(item['image'].convert('RGB'))
            label = item['label']
            
            batch_images.append(image)
            batch_labels.append(label)
            
            # Process batch when full or at the end
            if len(batch_images) == batch_size or i == num_samples - 1:
                # Stack batch
                batch_tensor = torch.stack(batch_images).to(device)
                batch_labels_tensor = torch.tensor(batch_labels).to(device)
                
                # Measure inference time
                start_inference = time.time()
                outputs = model(batch_tensor)
                end_inference = time.time()
                
                inference_times.append((end_inference - start_inference) / len(batch_images))
                
                # Calculate accuracy
                _, predicted = torch.max(outputs, 1)
                total += batch_labels_tensor.size(0)
                correct += (predicted == batch_labels_tensor).sum().item()
                
                # Reset batch
                batch_images = []
                batch_labels = []
    
    end_total = time.time()
    
    # Calculate metrics
    accuracy = 100 * correct / total
    avg_inference_time = sum(inference_times) / len(inference_times)
    total_time = end_total - start_total

    print(f"\nResults:")
    print(f"Accuracy: {accuracy:.2f}%")
    print(f"Avg Inference Time: {avg_inference_time*1000:.2f} ms")
    print(f"Total Evaluation Time: {total_time:.2f} seconds")
    print(f"Samples Processed: {total}")

In [5]:
import torchvision.models as models
import os

In [6]:
model_fp32 = models.resnet18(pretrained=True).eval()


/home/konrad/anaconda3/envs/de/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/konrad/anaconda3/envs/de/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [7]:
# Benchmark the FP32 model
benchmark(model_fp32, dataset, device=device)

# Benchmark and save
torch.save(model_fp32.state_dict(), "fp32.pth")
print("FP32 size (MB):", os.path.getsize("fp32.pth") / 1e6)

Evaluating model...


Processing: 100%|██████████| 1000/1000 [02:05<00:00,  7.99it/s]


Results:
Accuracy: 70.50%
Avg Inference Time: 0.13 ms
Total Evaluation Time: 125.33 seconds
Samples Processed: 1000
FP32 size (MB): 46.83002


# Pruning

Neural network pruning is a model compression technique that reduces the size and computational requirements of deep learning models by removing unnecessary parameters while maintaining performance. this addresses the problem where state-of-the-art models are often over-parametrized and contain redundancy that don't contribute to the model's predictive capabilities.

Pruning offers a solution by:
- Reducing model size
- Decreasing inference time
- Lowering energy consumption
- Enabling deployment on devices with limited resources

Pruning can achieve significant compression ratios while maintaining most of the original model's accuracy, making it an essential tool for model optimization and deployment.

The pruning process is as follows:
1. Train a full-precision model
2. Identify less important weights
3. Remove selected weights
4. (Optional, but recommended) Fine-tune the pruned model to recover any lost performance

## Unstructured Pruning

Unstructured pruning removes individual weights based on their magnitude or importance, regardless of their position in the network. This creates sparse weight matrices with irregular patterns of zeros.

In [8]:
import torch.nn as nn
import torch.nn.utils.prune as prune
import copy

In [9]:
model = copy.deepcopy(model_fp32)

# Apply unstructured pruning to all Conv2d and Linear layers
for name, module in model.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        prune.l1_unstructured(module, name='weight', amount=0.2)
        # Also prune bias if it exists
        if hasattr(module, 'bias') and module.bias is not None:
            prune.l1_unstructured(module, name='bias', amount=0.2)

for name, module in model.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        try:
            prune.remove(module, 'weight')
        except:
            pass
        try:
            prune.remove(module, 'bias')
        except:
            pass

benchmark(model, dataset, device=device)

# Save model
filename = f"unstructured_pruned_{int(50)}.pth"
torch.save(model.state_dict(), filename)
model_size = os.path.getsize(filename) / 1e6
print(f"  Model size: {model_size:.3f} MB")


Evaluating model...


Processing: 100%|██████████| 1000/1000 [01:18<00:00, 12.76it/s]


Results:
Accuracy: 69.60%
Avg Inference Time: 0.08 ms
Total Evaluation Time: 78.40 seconds
Samples Processed: 1000
  Model size: 46.839 MB


## Structured Pruning

Structured pruning removes entire structures like channels, filters, or layers, maintaining regular tensor shapes. While potentially less fine-grained than unstructured pruning, it's more hardware-friendly and can lead to actual speedups without specialized sparse computation libraries.

In [10]:
model = copy.deepcopy(model_fp32)

# Apply structured pruning to Conv2d layers
for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d):
        # Prune entire output channels
        num_channels = module.out_channels
        num_to_prune = int(num_channels * 0.2)
        
        if num_to_prune > 0:
            prune.ln_structured(
                module, 
                name='weight', 
                amount=num_to_prune, 
                n=2,  # L2 norm
                dim=0  # Output channel dimension
            )

# For Linear layers, use unstructured pruning
for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        prune.l1_unstructured(module, name='weight', amount=0.2)  # Less aggressive for FC layers

for name, module in model.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        try:
            prune.remove(module, 'weight')
        except:
            pass
        try:
            prune.remove(module, 'bias')
        except:
            pass

benchmark(model, dataset, device=device)

# Save model
filename = f"structured_pruned_{int(0.2*100)}.pth"
torch.save(model.state_dict(), filename)
model_size = os.path.getsize(filename) / 1e6
print(f"  Model size: {model_size:.3f} MB")

Evaluating model...


Processing: 100%|██████████| 1000/1000 [01:19<00:00, 12.60it/s]


Results:
Accuracy: 0.30%
Avg Inference Time: 0.08 ms
Total Evaluation Time: 79.43 seconds
Samples Processed: 1000
  Model size: 46.839 MB


## Global Pruning

Global pruning applies decision across the entire network simultaneously, allowing for optimal redistribution of parameters based on global importance metrics than layer-by-layer decisions.

In [11]:
model = copy.deepcopy(model_fp32)

# Collect all parameters to prune
parameters_to_prune = []
for name, module in model.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        parameters_to_prune.append((module, 'weight'))
        if hasattr(module, 'bias') and module.bias is not None:
            parameters_to_prune.append((module, 'bias'))

# Apply global unstructured pruning
prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=0.2,
)

for name, module in model.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        try:
            prune.remove(module, 'weight')
        except:
            pass
        try:
            prune.remove(module, 'bias')
        except:
            pass

benchmark(model, dataset, device=device)

# global pruning
filename = f"global_pruned_{int(0.2*100)}.pth"
torch.save(model.state_dict(), filename)
model_size = os.path.getsize(filename) / 1e6
print(f"  Model size: {model_size:.3f} MB")

Evaluating model...


Processing: 100%|██████████| 1000/1000 [01:50<00:00,  9.07it/s]


Results:
Accuracy: 70.20%
Avg Inference Time: 0.08 ms
Total Evaluation Time: 110.24 seconds
Samples Processed: 1000
  Model size: 46.838 MB


Local Pruning:
- Makes pruning decisions layer-by-layer or module-by-module
- Each layer gets pruned independently (e.g., "remove 20% of weights from each layer")
- Simpler to implement but potentially suboptimal

Global Pruning:
- Makes pruning decisions across the entire network
- Compares importance of weights from all layers simultaneously
- Allows optimal redistribution of parameters based on global importance

Global pruning often performs better because it can intelligently decide that some layers need more parameters than others, rather than uniformly reducing all layers by the same percentage.

| Combination | Description |
|-------------|-------------|
| Global + Unstructured | Compare all individual weights globally, remove the least important ones network-wide |
| Local + Unstructured | Remove individual weights within each layer independently |
| Global + Structured | Compare importance of entire structures globally |
| Local + Structured | Remove structures within each layer independently |

# Quantization

Neural network quantization is a model compression technique that reduces the precision of weights and activations from the standard 32-bit floating-point (FP32) representation to lower-precision formats such as 8-bit integers (INT8) or even lower. This change in numerical representation can dramatically reduce model size and accelerate inference.

While standard FP32 arithmetic provides high precision, it comes with significant computational and memory overhead, Quantization addresses it by:
* Reducing memory footprint
* Accelerating inference through faster integer arithmetic
* Enabling deployment on devices with limited compute
* Lowering bandwidth requirements
* Decreasing power consumption

Types of quantization:
1. Post-Training Quantization (PTQ)
2. Quantization-Awate Training (QAT)

Linear Quantization formula:
```
quantized_value = round((float_value - zero_point) / scale)
dequantized_value = scale * (quantized_value + zero_point)
```

Where:
- Scale: Controls the range mapping between float and integer domains
- Zero-point: Ensures exact representation of zero

## Dynamic Quantization

Dynamic Quantization, belonging to the family of PTQ techniques, quantizes weights offline but keeps activations in FP32. It converts activations to INT8 dynamically during inference and offers good balance of speed and accuracy with minimal setup required. Best for models with frequent memory access, such as RNNs and Transformers.

In [12]:
model_fp32_cpu = copy.deepcopy(model_fp32).to('cpu')
benchmark(model_fp32_cpu, dataset)

Evaluating model...


Processing: 100%|██████████| 1000/1000 [01:51<00:00,  8.99it/s]


Results:
Accuracy: 70.50%
Avg Inference Time: 14.49 ms
Total Evaluation Time: 111.48 seconds
Samples Processed: 1000


In [13]:
from torch.quantization import quantize_dynamic

model_dynamic = copy.deepcopy(model_fp32_cpu)
model_dynamic = quantize_dynamic(model_dynamic, qconfig_spec={nn.Linear, nn.Conv2d}, dtype=torch.qint8)

# Benchmark the dynamic quantized model
benchmark(model_dynamic, dataset)

torch.save(model_dynamic.state_dict(), "dynamic.pth")
print("Dynamic Quantized size (MB):", os.path.getsize("dynamic.pth") / 1e6)


Evaluating model...


Processing: 100%|██████████| 1000/1000 [01:13<00:00, 13.52it/s]



Results:
Accuracy: 70.20%
Avg Inference Time: 13.67 ms
Total Evaluation Time: 74.15 seconds
Samples Processed: 1000
Dynamic Quantized size (MB): 45.299578


## Static Quantization

Static Quantization, post-training quantization technique, quantizes both weights and activations to INT8, offers maximum compression and speedup, but requires calibration dataset to determine optimal quantization parameters. Best for convolutional networks and for production deployment.

In [14]:
from torch.ao.quantization import get_default_qconfig, fuse_modules, prepare, convert
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx

In [15]:
example_inputs = torch.randn(1, 3, 224, 224)
        
model = copy.deepcopy(model_fp32_cpu)
model.eval()

# Configure quantization
qconfig_dict = {"": get_default_qconfig('fbgemm')}

model_prepared = prepare_fx(model, qconfig_dict, example_inputs=example_inputs)

calibration_samples = 0
max_calibration_samples = 100  # Fewer samples for FX

with torch.no_grad():
    for i, item in enumerate(dataset):
        if calibration_samples >= max_calibration_samples:
            break
        
        transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                std=[0.229, 0.224, 0.225])
        ])
        
        image = transform(item['image'].convert('RGB'))
        image = image.unsqueeze(0)
        
        model_prepared(image)
        calibration_samples += 1
            
# Convert
model_quantized_fx = convert_fx(model_prepared)

# Benchmark the static quantized model
benchmark(model_quantized_fx, dataset)

# Save and benchmark
torch.save(model_quantized_fx.state_dict(), "static_fx.pth")
fx_size = os.path.getsize("static_fx.pth") / 1e6
print(f"Static Quantized Model size: {fx_size:.3f} MB")
        


/home/konrad/anaconda3/envs/de/lib/python3.10/site-packages/torch/ao/quantization/quantize_fx.py:146: FutureWarning: Passing a QConfig dictionary to prepare is deprecated and will not be supported in a future version. Please pass in a QConfigMapping instead.
  prepared = prepare(
/home/konrad/anaconda3/envs/de/lib/python3.10/site-packages/torch/ao/quantization/observer.py:229: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(


Evaluating model...


Processing: 100%|██████████| 1000/1000 [01:06<00:00, 15.09it/s]


Results:
Accuracy: 70.90%
Avg Inference Time: 5.14 ms
Total Evaluation Time: 66.35 seconds
Samples Processed: 1000
Static Quantized Model size: 11.838 MB


## Quantization Aware Training (QAT)
`note: this introduction was heavily inspired by the official PyTorch documentation on quantization`
#### Overview:
Quantization Aware Training (QAT) simulates the effects of quantization during training, allowing a neural network to learn to be robust to quantization noise. Unlike post-training quantization, QAT can maintain almost the same accuracy as full precision models — especially for convolutional neural networks (CNNs) and edge-deployed models.

#### Motivation:
Post-training quantization can cause significant accuracy drops for some models (especially CNNs).
QAT fixes this by training with quantization noise from the start.

#### Mechanism:
    graph TD
    A[Float32 Model] --> B[Insert FakeQuant Modules] 
    B --> C[Train with Quantization Noise]
    C --> D[Convert to INT8 Model]
    D --> E[Deploy on Edge Device]

QAT as pseudocode
```python
# define a floating point model where some layers could benefit from QAT
class M(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # QuantStub converts tensors from floating point to quantized
        self.quant = torch.ao.quantization.QuantStub()
        self.conv = torch.nn.Conv2d(1, 1, 1)
        self.bn = torch.nn.BatchNorm2d(1)
        self.relu = torch.nn.ReLU()
        # DeQuantStub converts tensors from quantized to floating point
        self.dequant = torch.ao.quantization.DeQuantStub()

    def forward(self, x):
        x = self.quant(x)
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.dequant(x)
        return x

# create a model instance
model_fp32 = M()

# model must be set to eval for fusion to work
model_fp32.eval()

# attach a global qconfig, which contains information about what kind
# of observers to attach. Use 'x86' for server inference and 'qnnpack'
# for mobile inference. Other quantization configurations such as selecting
# symmetric or asymmetric quantization and MinMax or L2Norm calibration techniques
# can be specified here.
# Note: the old 'fbgemm' is still available but 'x86' is the recommended default
# for server inference.
# model_fp32.qconfig = torch.ao.quantization.get_default_qconfig('fbgemm')
model_fp32.qconfig = torch.ao.quantization.get_default_qat_qconfig('x86')

# fuse the activations to preceding layers, where applicable
# this needs to be done manually depending on the model architecture
model_fp32_fused = torch.ao.quantization.fuse_modules(model_fp32,
    [['conv', 'bn', 'relu']])

# Prepare the model for QAT. This inserts observers and fake_quants in
# the model needs to be set to train for QAT logic to work
# the model that will observe weight and activation tensors during calibration.
model_fp32_prepared = torch.ao.quantization.prepare_qat(model_fp32_fused.train())

# run the training loop (not shown)
training_loop(model_fp32_prepared)

# Convert the observed model to a quantized model. This does several things:
# quantizes the weights, computes and stores the scale and bias value to be
# used with each activation tensor, fuses modules where appropriate,
# and replaces key operators with quantized implementations.
model_fp32_prepared.eval()
model_int8 = torch.ao.quantization.convert(model_fp32_prepared)

# run the model, relevant calculations will happen in int8
res = model_int8(input_fp32)
```

## Knowledge(Model) Distillation
`note: this introduction was heavily inspired by the official PyTorch documentation on knowledge distillation`
#### Overview:
Knowledge Distillation is a model compression technique where a smaller model (the student) learns to mimic a larger, more powerful model (the teacher). The student model is trained not just on the ground truth labels, but also on the soft targets (probability distributions) produced by the teacher model.

#### Motivation:
Larger models (e.g., Transformers, ResNets) often have excellent performance, but are too slow or large for deployment.
Knowledge Distillation helps transfer their knowledge to smaller, faster models without substantial accuracy loss.

#### Mechanism:
    graph TD
    A[Input Data] --> B[Teacher Model (Pretrained)]
    A --> C[Student Model (Trainable)]
    B --> D[Soft Targets (High-T Softmax)]
    C --> E[Student Predictions (High-T)]
    D --> F[KL Loss]
    E --> F
    F --> G[Total Loss (with CE)]

model distillation as pseudocode
```python
def distillation_loss(student_logits, teacher_logits, labels, T=2.0, alpha=0.5):
    # Standard cross-entropy loss with ground truth
    ce_loss = F.cross_entropy(student_logits, labels)
    
    # Softened probabilities
    student_probs = F.log_softmax(student_logits / T, dim=1)
    teacher_probs = F.softmax(teacher_logits / T, dim=1)
    
    # KL divergence between softened outputs
    kd_loss = F.kl_div(student_probs, teacher_probs, reduction='batchmean') * (T * T)
    
    return alpha * ce_loss + (1 - alpha) * kd_loss


teacher = ResNet50(pretrained=True).eval()  # Larger, accurate
student = ResNet18()                        # Smaller, faster

for batch in dataloader:
    inputs, labels = batch
    with torch.no_grad():
        teacher_outputs = teacher(inputs)
    student_outputs = student(inputs)
    
    loss = distillation_loss(student_outputs, teacher_outputs, labels)
    loss.backward()
    optimizer.step()
```

Practical Considerations:
- INT8 quantization is the most common target, offering good balance
- Calibration data should be representative of real inference data
- Symmetric vs asymmetric quantization affects zero-point handling
- Per-channel quantization can improve accuracy for convolutions
- Mixed precision allows keeping sensitive layers in higher precision

Quantization is often the first optimization technique applied in production environments due to its ease of implementation and significant benefits, making it an essential tool for deploying neural networks at scale.

## ONNX

ONNX (Open Neural Network Exchange) is an open-source format that provides a standard representation for machine learning models, enabling interoperability between different deep learning frameworks. ONNX defines a common set of operators and a standard file format that allows models to be represented in a framework-agnostic way. It acts as an intermediate representation that captures the computational graph of neural networks.

ONNX offers:
- Cross-platform Deployment - train in PyTorch/TensorFlow and deploy anywhere
- Performance Optimization - graph optimization, hardware-specific accelerations
- Production-Ready inference - standarized format, reduced dependencies

In [16]:
benchmark(model, dataset, batch_size=1, device=device)

Evaluating model...


Processing: 100%|██████████| 1000/1000 [01:03<00:00, 15.64it/s]


Results:
Accuracy: 70.50%
Avg Inference Time: 2.29 ms
Total Evaluation Time: 63.96 seconds
Samples Processed: 1000


In [17]:
import onnx
import onnxruntime as ort
import numpy as np
from pathlib import Path

In [18]:
model = copy.deepcopy(model_fp32).to(device)
model.eval()
    
# Create dummy input
dummy_input = torch.randn((1, 3, 224, 224)).to(device)

# Define input and output names
input_names = ['input']
output_names = ['output']

# Convert to ONNX
onnx_path = "model.onnx"

torch.onnx.export(
    model,                          # Model to export
    dummy_input,                    # Model input (or a tuple for multiple inputs)
    onnx_path,                      # Where to save the model
    export_params=True,             # Store the trained parameter weights inside the model file
    opset_version=11,               # ONNX version to export the model to
    do_constant_folding=True,       # Whether to execute constant folding for optimization
    input_names=input_names,        # Model's input names
    output_names=output_names,      # Model's output names
    dynamic_axes={
        'input': {0: 'batch_size'},     # Variable length axes
        'output': {0: 'batch_size'}
    }
)

# Check the model
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print("✓ ONNX model validation passed")

# Get model size
onnx_size = Path(onnx_path).stat().st_size / 1e6
print(f"✓ ONNX model size: {onnx_size:.3f} MB")

✓ ONNX model validation passed
✓ ONNX model size: 46.749 MB


In [19]:
def benchmark_onnx(onnx_path, input_shape=(1, 3, 224, 224), num_runs=100):
    available_providers = ort.get_available_providers()
    if 'CUDAExecutionProvider' in available_providers:
        providers = [('CUDAExecutionProvider', {
            'device_id': 0,
            'arena_extend_strategy': 'kNextPowerOfTwo',
            'gpu_mem_limit': 5 * 1024 * 1024 * 1024,  # 2GB limit
            'cudnn_conv_algo_search': 'EXHAUSTIVE',
            'do_copy_in_default_stream': True,
            'cudnn_conv_use_max_workspace': '1',
            'enable_cuda_graph': True,
        })]
    else:
        providers = ['CPUExecutionProvider']
    print(f"Using providers: {providers}")

    session = ort.InferenceSession(onnx_path, providers=providers)

    # Get input and output details
    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name
    input_shape_info = session.get_inputs()[0].shape
    output_shape_info = session.get_outputs()[0].shape

    print(f"Input name: {input_name}")
    print(f"Output name: {output_name}")
    print(f"Input shape: {input_shape_info}")
    print(f"Output shape: {output_shape_info}")

    io_binding = session.io_binding()
    
    # Allocate GPU memory for input
    input_tensor = torch.randn(input_shape, dtype=torch.float32, device='cuda')
    
    # Calculate output shape (replace 'batch_size' with actual batch size)
    output_shape = list(output_shape_info)
    if 'batch_size' in str(output_shape):
        output_shape = [input_shape[0], 1000]  # ResNet18 has 1000 output classes
    
    # Allocate GPU memory for output
    output_tensor = torch.empty(output_shape, dtype=torch.float32, device='cuda')
    
    # Bind input and output to GPU memory
    io_binding.bind_input(
        name=input_name,
        device_type=device,
        device_id=0,
        element_type=np.float32,
        shape=input_tensor.shape,
        buffer_ptr=input_tensor.data_ptr()
    )
    
    io_binding.bind_output(
        name=output_name,
        device_type=device,
        device_id=0,
        element_type=np.float32,
        shape=output_tensor.shape,
        buffer_ptr=output_tensor.data_ptr()
    )
    
    print(f"✓ IOBinding configured for GPU tensors")
    print(f"  Input tensor shape: {input_tensor.shape}")
    print(f"  Output tensor shape: {output_tensor.shape}")
    
    # Warmup runs with IOBinding
    print(f"\nRunning {10} warmup iterations with IOBinding...")
    for _ in range(10):
        # Update input data (simulate new batch)
        input_tensor.random_()
        session.run_with_iobinding(io_binding)
        torch.cuda.synchronize()  # Ensure GPU operations complete
    
    # Timing runs with IOBinding
    print(f"Running {num_runs} timing iterations with IOBinding...")
    torch.cuda.synchronize()
    start_time = time.time()
    
    for _ in range(num_runs):
        # Update input data
        input_tensor.random_()
        session.run_with_iobinding(io_binding)
    
    torch.cuda.synchronize()
    end_time = time.time()

    # Calculate metrics
    total_time = end_time - start_time
    avg_time = (total_time / num_runs) * 1000  # Convert to ms

    print(f"- Average inference time: {avg_time:.3f} ms")
    print(f"- Total time for {num_runs} runs: {total_time:.3f} seconds")
    print(f"- Throughput: {num_runs / total_time:.1f} inferences/second")

In [20]:
benchmark_onnx("model.onnx", (1, 3, 224, 224), 1000)

Using providers: [('CUDAExecutionProvider', {'device_id': 0, 'arena_extend_strategy': 'kNextPowerOfTwo', 'gpu_mem_limit': 5368709120, 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': True, 'cudnn_conv_use_max_workspace': '1', 'enable_cuda_graph': True})]
Input name: input
Output name: output
Input shape: ['batch_size', 3, 224, 224]
Output shape: ['batch_size', 1000]
✓ IOBinding configured for GPU tensors
  Input tensor shape: torch.Size([1, 3, 224, 224])
  Output tensor shape: torch.Size([1, 1000])

Running 10 warmup iterations with IOBinding...
Running 1000 timing iterations with IOBinding...
- Average inference time: 2.182 ms
- Total time for 1000 runs: 2.182 seconds
- Throughput: 458.3 inferences/second


In [21]:
# Create session options
sess_options = ort.SessionOptions()
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED
sess_options.optimized_model_filepath = f"optimized_{Path(onnx_path).name}"

print(f"Optimized model path: {sess_options.optimized_model_filepath}")

sess_options.execution_mode = ort.ExecutionMode.ORT_PARALLEL
sess_options.inter_op_num_threads = 0  # Use all available cores
sess_options.intra_op_num_threads = 0  # Use all available cores

# Create session (this will generate optimized model)
session = ort.InferenceSession(onnx_path, sess_options)

# Get optimized model size
if Path(sess_options.optimized_model_filepath).exists():
    opt_size = Path(sess_options.optimized_model_filepath).stat().st_size / 1e6
    print(f"✓ Optimized model size: {opt_size:.3f} MB")

Optimized model path: optimized_model.onnx
✓ Optimized model size: 46.749 MB


In [22]:
benchmark_onnx("optimized_model.onnx", (1, 3, 224, 224), 1000)

Using providers: [('CUDAExecutionProvider', {'device_id': 0, 'arena_extend_strategy': 'kNextPowerOfTwo', 'gpu_mem_limit': 5368709120, 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': True, 'cudnn_conv_use_max_workspace': '1', 'enable_cuda_graph': True})]
Input name: input
Output name: output
Input shape: ['batch_size', 3, 224, 224]
Output shape: ['batch_size', 1000]
✓ IOBinding configured for GPU tensors
  Input tensor shape: torch.Size([1, 3, 224, 224])
  Output tensor shape: torch.Size([1, 1000])

Running 10 warmup iterations with IOBinding...
Running 1000 timing iterations with IOBinding...
- Average inference time: 2.103 ms
- Total time for 1000 runs: 2.103 seconds
- Throughput: 475.5 inferences/second


## Summary

| Optimization Technique | Accuracy (%) | Avg Inference Time (ms) | Model Size (MB) | Notes |
|------------------------|--------------|-------------------------|-----------------|-------|
| Baseline FP32 (GPU) | 70.50 | 0.13 | 46.83 | Batch size = 32 |
| Unstructured Pruning (20%) | 69.60 | 0.08 | 46.84 | Individual weight removal |
| Structured Pruning (20%) | 0.30 | 0.08 | 46.84 | Accuracy degradation |
| Global Pruning (20%) | 70.20 | 0.08 | 46.84 | Best pruning approach |
| |
| Baseline FP32 (CPU) | 70.50 | 14.49 | 46.83 | Same model on CPU |
| Dynamic Quantization | 70.20 | 13.67 | 45.30 | INT8, Linear and Conv2d layers |
| Static Quantization | 70.90 | 5.14 | 11.84 | INT8, Best compression ratio |
| |
| Baseline FP32 (GPU) | - | 2.29 | - | Batch size = 1 |
| ONNX Export | - | 2.182 | 46.75 | Cross-platform deployment |
| Optimized ONNX | - | 2.103 | 46.75 | Minimal improvement |